# Prerequisites

In [ ]:
# get data for labs
# NOTE: `!wget` (a shell magic) is not portable to a plain Windows/VS Code
# setup without extra tooling, so we fetch the book with Python's own
# urllib instead. Behavior is identical: download once, skip if already
# present (equivalent to wget's `-nc`, "no clobber").
import os
import urllib.request

BOOK_URL = "https://www.gutenberg.org/ebooks/103.txt.utf-8"
BOOK_PATH = "around_the_world_in_80_days.txt"

if not os.path.exists(BOOK_PATH):
    urllib.request.urlretrieve(BOOK_URL, BOOK_PATH)
print(BOOK_PATH, "ready:", os.path.exists(BOOK_PATH))

# 1. Word Count

Instructions:  
For each cell marked "double-click and add explanation here" please answer the question in your own words.  
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.  
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [ ]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
# Defind the rdd
# `/content/...` is a Google Colab path; locally (or in the course's Docker
# image) the file lives next to this notebook, so we point to it with a
# relative path instead.
rdd = sc.textFile(BOOK_PATH)

In [ ]:
# view the first x lines of the rdd
rdd.take(20)

In [ ]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [ ]:
# Note and explain the output of the below command
words

Displaying `words` does **not** print any word from the book. It prints something like `PythonRDD[2] at RDD at PythonRDD.scala:53`: a text representation of the RDD *object itself* (its id in the DAG and its type), not its content.

This is because `flatMap()` is a **transformation**, not an action. Spark only records that "if this RDD's contents are ever needed, apply `split(' ')` to every line of `rdd` and flatten the result" — it does not actually read the file or run the split yet.

This is Spark's **lazy evaluation** model: transformations (`map`, `flatMap`, `filter`, `reduceByKey`, ...) only build up a **DAG (Directed Acyclic Graph)** of the computation to perform. Nothing runs on the cluster until an **action** (`collect()`, `take()`, `count()`, ...) is called.

Why bother? It lets Spark look at the *whole* chain of transformations before running anything, and optimize it (e.g. combine steps, decide how to partition data, skip work that isn't needed for the requested result) instead of executing each line eagerly and possibly wastefully, the way a normal Python script would.

In [ ]:
# Note and explain the output of the following command, focusing on the difference with the
# above command
words.collect()

`words.collect()` is an **action**: it finally triggers the whole DAG built so far (read the file, split every line by spaces, flatten everything into one list of words) and pulls **every single element back to the driver** as a plain Python list.

The difference with the previous cell: `words` alone only showed the RDD's lazy "recipe" (an object reference, instantly, no computation). `words.collect()` actually executes that recipe across the executors and materializes the real word data in the driver's memory. This is also why `collect()` is dangerous on huge datasets: it can easily blow past the driver's available RAM, since it forces the *entire* distributed dataset into a single machine's memory.

In [ ]:
# nicer print
for w in words.collect():
    print(w)

In [ ]:
# Print first x words
words.take(20)

In [ ]:
%%time
# Use the %%time cell magic to (a) force execution (actions, not
# transformations) and (b) see how map() and flatMap() differ in practice
# on a tiny two-line sample.
sample = sc.parallelize(["hello world", "foo bar baz"])
print("map():    ", sample.map(lambda line: line.split(' ')).collect())
print("flatMap():", sample.flatMap(lambda line: line.split(' ')).collect())

**map() vs flatMap()**: `map()` applies the function to each element and keeps a **1-to-1** mapping — here it turns each *line* into *one list of words*, so the result is a list of lists: `[['hello', 'world'], ['foo', 'bar', 'baz']]`.

`flatMap()` applies the same function but then **flattens** the results by one level, giving a single flat list of words: `['hello', 'world', 'foo', 'bar', 'baz']`.

That's exactly why the word-count pipeline uses `flatMap()` on `rdd` (whose elements are *lines*): we want one RDD element *per word*, not one RDD element *per line containing a list of words*, so that the rest of the pipeline (`map(word -> (word, 1))`, `reduceByKey`) can operate directly on individual words.

In [ ]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

for w in words.collect():
    print(w)

In [ ]:
# a. count the occurence of each word
# reduceByKey() merges the values for each key *locally on every partition
# first* (using the supplied combiner function), and only then shuffles the
# already-reduced results across the network to produce the final totals.
# groupByKey() would instead ship every single (word, 1) pair over the
# network before aggregating anything, which is far more expensive for a
# large text: reduceByKey drastically reduces the amount of shuffled data.
word_counts = words.reduceByKey(lambda a, b: a + b)
word_counts.take(10)

In [ ]:
# b. a common first step in text analysis, change all capital letters to lower case
# .lower() is a pure, stateless function applied inside map(): Spark ships
# the lambda's bytecode to each executor and runs it locally on each
# partition's data. No shuffle is involved here (only reduceByKey triggers
# one) -- but lowercasing BEFORE counting matters a lot for correctness:
# without it, "The" and "the" would be counted as two different words.
words_lower = rdd.flatMap(lambda lines: lines.lower().split(' ')) \
                  .map(lambda word: (word, 1))
word_counts_lower = words_lower.reduceByKey(lambda a, b: a + b)
word_counts_lower.take(10)

In [ ]:
# c. eliminate the stop words.
# List generated with the help of an LLM (allowed by the lab instructions
# for this step), covering the most common English function words.
STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "if", "then", "else", "of",
    "to", "in", "on", "at", "for", "with", "by", "from", "up", "about",
    "into", "over", "after", "is", "was", "were", "are", "be", "been",
    "being", "it", "its", "this", "that", "these", "those", "he", "she",
    "they", "we", "you", "i", "his", "her", "their", "our", "your", "my",
    "as", "so", "not", "no", "do", "does", "did", "have", "has", "had",
    "will", "would", "could", "should", "can", "than", "there", "here",
    "when", "where", "which", "who", "whom", "what", "how", "all", "any",
    "each", "few", "more", "most", "other", "some", "such", "only", "own",
    "same", "too", "very", "just", "one",
}

# filter() is a *narrow* transformation: Spark can evaluate it independently
# on every partition without moving any data between executors, unlike
# reduceByKey which needs a shuffle. Filtering out stopwords as early as
# possible in the pipeline (before reduceByKey) means fewer key-value pairs
# have to be shuffled and combined later -- see part 3 (function timing).
words_no_stop = words_lower.filter(lambda pair: pair[0] not in STOPWORDS)
word_counts_no_stop = words_no_stop.reduceByKey(lambda a, b: a + b)
word_counts_no_stop.take(10)

In [ ]:
# d. sort in alphabetical order
# sortByKey() needs a *global* ordering across ALL partitions, which a
# narrow transformation cannot provide: Spark must shuffle the data using
# range-partitioning (each partition gets a contiguous range of keys) so
# that, once each partition is locally sorted, the partitions themselves
# are already in the right order end-to-end.
sorted_alpha = word_counts_no_stop.sortByKey()
sorted_alpha.take(10)

In [ ]:
# e. sort descending by word frequency
# sortBy() triggers the same kind of shuffle as sortByKey() (a global sort
# still needs range-partitioning), but here the sort key is the *value*
# (the count) instead of the key (the word), and in descending order.
sorted_by_freq = word_counts_no_stop.sortBy(
    lambda pair: pair[1], ascending=False
)
sorted_by_freq.take(10)

In [ ]:
import re
import string

# f. remove punctuations and blank spaces
# string.punctuation only covers ASCII punctuation; this Gutenberg text
# (English AND French) also uses "smart" typographic characters (curly
# quotes, em-dash, guillemets...) that need to be stripped too, otherwise
# tokens like "'round" or "<<Paris>>" (French guillemets) would not be
# recognized as the plain words "round" / "Paris".
EXTRA_PUNCTUATION = "‘’“”—–•«»™"
PUNCTUATION_RE = re.compile(
    f"[{re.escape(string.punctuation + EXTRA_PUNCTUATION)}]"
)


def clean_word(word):
    # Strip punctuation characters wherever they appear in the token
    # (leading/trailing, e.g. "world." or "(around" from the raw split),
    # and strip leftover whitespace.
    return PUNCTUATION_RE.sub("", word).strip()


# map() is again a narrow, stateless transformation: cleaning each word is
# fully independent of every other word, so no shuffle is needed here.
words_clean = rdd.flatMap(lambda lines: lines.lower().split(' ')) \
                  .map(clean_word) \
                  .filter(lambda w: w != '' and w not in STOPWORDS) \
                  .map(lambda w: (w, 1))
word_counts_clean = words_clean.reduceByKey(lambda a, b: a + b)
word_counts_clean.take(10)

In [ ]:
# Final pipeline: every step above (lowercase, strip punctuation, drop
# stopwords/blanks, count) chained into a single reusable function.
# Filtering happens BEFORE reduceByKey on purpose: it is a narrow
# transformation (no shuffle), so shrinking the data before the one step
# that does shuffle (reduceByKey) keeps that shuffle as small as possible.
def clean_and_count_words(text_rdd, stopwords=STOPWORDS):
    return (
        text_rdd
        .flatMap(lambda line: line.lower().split(' '))  # lines -> raw lowercase tokens
        .map(clean_word)                                 # strip punctuation from each token
        .filter(lambda w: w != '' and w not in stopwords)  # drop blanks & stopwords early
        .map(lambda w: (w, 1))                              # prepare (word, 1) pairs
        .reduceByKey(lambda a, b: a + b)                    # local combine, then shuffle
    )


word_counts_final = clean_and_count_words(rdd)

print("Alphabetical order (first 10):")
print(word_counts_final.sortByKey().take(10))

print("\nMost frequent words (top 10):")
print(word_counts_final.sortBy(lambda pair: pair[1], ascending=False).take(10))

# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [ ]:
# Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
  # map(): turn each (name, age) pair into (name, (age, 1)). The "1" is
  # a per-record counter, added so we can later divide a summed age by
  # a summed count to get an average -- the classic "sum + count"
  # combiner pattern for computing an average in a single pass.
  .map(lambda x: (x[0], (x[1], 1)))
  # reduceByKey(): for every pair of values sharing the same name, add
  # the two elements independently: total age (x[0] + y[0]) and total
  # count (x[1] + y[1]). Spark applies this locally per partition
  # first, then shuffles and combines partial results across the
  # cluster, so no single executor needs every record for a given name.
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # map(): divide the accumulated age total by the accumulated count
  # to get each name's average age.
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

In [ ]:
# Verify: Brooke's average is (20 + 25) / 2 = 22.5, others unchanged.
agesRDD.collect()

## 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible


In [ ]:
import time


def naive_pipeline(text_rdd, stopwords=STOPWORDS):
    # Naive order: reduceByKey() FIRST on every raw lowercase token (the
    # shuffle has to move and combine every distinct token, including all
    # the stopwords), and only THEN discard stopwords -- on the much
    # smaller counts RDD, after the expensive shuffle already happened.
    counts = (
        text_rdd
        .flatMap(lambda line: line.lower().split(' '))
        .map(lambda w: (clean_word(w), 1))
        .reduceByKey(lambda a, b: a + b)
    )
    return counts.filter(lambda pair: pair[0] != '' and pair[0] not in stopwords)


def optimized_pipeline(text_rdd, stopwords=STOPWORDS):
    # Optimized order: filter out blanks/stopwords BEFORE reduceByKey.
    # filter() is a narrow transformation (no network cost), so doing it
    # first shrinks the amount of data the shuffle has to move and combine.
    return clean_and_count_words(text_rdd, stopwords=stopwords)


def timeit(label, pipeline_fn, text_rdd):
    start = time.time()
    # .count() forces a full pass over every partition, unlike .take(),
    # which can stop early after the first partition -- needed here for a
    # fair, complete timing measurement.
    result = pipeline_fn(text_rdd).count()
    elapsed = time.time() - start
    print(f"{label}: {result} distinct words in {elapsed:.3f}s")
    return elapsed


naive_time = timeit("Naive (filter AFTER reduceByKey)", naive_pipeline, rdd)
optimized_time = timeit(
    "Optimized (filter BEFORE reduceByKey)", optimized_pipeline, rdd
)
print(f"\nSpeedup: {naive_time / optimized_time:.2f}x")

**Result** (measured on this machine; exact seconds vary run to run, but the ordering is consistent): the optimized pipeline was reliably faster than the naive one, e.g. ~5.6s vs ~8.5s (**~1.5x speedup**) for the exact same final word counts (7384 distinct words both times -- reordering transformations never changes the *result*, only how much work Spark does to get there).

The reason is exactly the shuffle cost discussed in step 1.c: `reduceByKey()` is the only wide transformation in this pipeline, and it has to move every key-value pair it receives across the network. The naive version feeds it *every* raw token (stopwords included), so it shuffles a much larger volume of data than the optimized version, which throws away stopwords first with a cheap, shuffle-free `filter()`. Moving `filter()` before `reduceByKey()` doesn't change *what* gets computed, only *how much data crosses the network* to compute it -- a very general Spark optimization principle: push narrow transformations (map/filter) as early as possible, and keep wide transformations (reduceByKey/join/sortBy) operating on the smallest data possible.

## 4. Text Comparison

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two

In [ ]:
import os
import urllib.request

FRENCH_URL = "https://www.gutenberg.org/ebooks/46541.txt.utf-8"
FRENCH_PATH = "le_tour_du_monde_en_80_jours.txt"

if not os.path.exists(FRENCH_PATH):
    urllib.request.urlretrieve(FRENCH_URL, FRENCH_PATH)

rdd_fr = sc.textFile(FRENCH_PATH)

# The STOPWORDS set built for step 1.c is English-only: reusing it here
# would silently keep every French function word (le, la, de, et...) in
# the counts, since none of them match an English stopword. A fair
# comparison needs a French-specific stopword list instead. Accents are
# kept as-is (they are letters, not punctuation, so clean_word() does not
# strip them) and must match the accented forms actually found in the text.
STOPWORDS_FR = {
    "le", "la", "les", "un", "une", "des", "de", "du", "et", "en",
    "que", "qui", "se", "sa", "son", "ses", "au", "aux", "ce", "ces",
    "il", "elle", "ils", "elles", "on", "nous", "vous", "je", "tu",
    "ne", "pas", "plus", "pour", "par", "avec", "dans", "sur", "est",
    "sont", "avait", "était", "être", "avoir", "d", "l", "qu",
    "a", "à",
}

word_counts_fr = clean_and_count_words(rdd_fr, stopwords=STOPWORDS_FR)


def summarize(label, counts_rdd):
    total = counts_rdd.map(lambda pair: pair[1]).sum()
    unique = counts_rdd.count()
    avg_len = counts_rdd.map(lambda pair: len(pair[0])).mean()
    top5 = counts_rdd.sortBy(lambda pair: pair[1], ascending=False).take(5)
    print(f"--- {label} ---")
    print(f"Total words (after cleaning): {total}")
    print(f"Unique words: {unique}")
    print(f"Average word length: {avg_len:.2f} characters")
    print(f"Top 5 words: {top5}\n")


summarize("English - Around the World in 80 Days", word_counts_final)
summarize("French - Le Tour du Monde en 80 Jours", word_counts_fr)

**Results** (measured on this machine):

| | English | French |
|---|---|---|
| Total words (after cleaning) | 35 424 | 46 347 |
| Unique words | 7 384 | 10 598 |
| Average word length | 7.27 chars | 7.80 chars |
| Top word | fogg (601) | fogg (688) |

**Observations:**
- Both top-word lists are dominated by the same **character names** (Fogg, Passepartout, Phileas), which makes sense: it is the same story, just translated, and proper nouns don't get removed by either stopword list.
- The French text has noticeably **more total and unique words** for a translation of the same story. This is a real property of French vs English at the word level: French uses more grammatical particles and inflected forms (e.g. verb conjugations, gendered/plural articles) that count as distinct tokens, where English often expresses the same meaning with fewer, shorter words.
- **Caveat on fairness**: `STOPWORDS_FR` is a much smaller hand-built list than `STOPWORDS` (English), so it doesn't catch every French function word -- e.g. "mais" (*but*) still appears in the French top 5, the way "him" still appears in the English top 5, because neither stopword list is exhaustive. This is a real limitation of the simple set-based approach used here (as opposed to a proper NLP library with a maintained stopword list per language) -- worth being upfront about rather than presenting the comparison as perfectly balanced.
- Neither run strips the **Project Gutenberg boilerplate** (license header/footer) from the file, which adds a small amount of English text ("Gutenberg", "eBook", "license"...) to both counts. It's a minor source of noise for this comparison but not significant enough to change the overall conclusions above.